In [ ]:
#This code is used if the notebook is implemented in github codespace. Just remove the (#)
!python -m pip install .. --quiet

In [ ]:
# import sys
# print(sys.path)  # check that project root or src is in path
# import luma_ge
# print(luma_ge.__file__)

In [ ]:
# %load_ext autoreload
# %autoreload 2

# Import GEE account setting from luma-stack

In [ ]:
# Use EE initialization from luma_ge
import ee 
import luma_ge

# Autheticate using service account (json file)

service_account_path = '../auth/ee-epstm2024.json'
success = luma_ge.initialize_with_service_account(service_account_path)

if success:
    print("Earth Engine initialized with service account successfully!")
else:
    print("Service account initialization failed. Try to authenticate earth engine manually")

#Check authentication status
status = luma_ge.get_auth_status()
print(f"Initialized: {status['initialized']}")
print(f"Authenticated: {status['authenticated']}")
if status['project']:
    print(f"Project: {status['project']}")


# Upload AOI

In [ ]:
# Upload AOI

import geemap
from luma_ge.data_acquisition import Reflectance_Data, Reflectance_Stats, final_Image
import geopandas as gpd

aoi = geemap.shp_to_ee("../data/test_data/area_of_interest.shp")

# Collect satellite images

In [ ]:
#========== FIRST RETRIVE THE MULTISPECTRAL BAND===========
#Intialize the relfectance class data function
optical_reflectance = Reflectance_Data()
#Initialize the final image class for composite creation
composite = final_Image() #NEW FEATURE ADDED HERE
#define the start and end date for imagery collection
start = '2024-01-01'
end = '2024-12-31'
#get the image collection and corresponding statistics
landsat_data, meta = optical_reflectance.get_optical_data(aoi, start, end, optical_data='L8_SR', 
                                                           cloud_cover=40, compute_detailed_stats=False)
#create mosaic between image collection, and clip based on AOI
mosaic_landsat = composite.get_quality_mosaic(landsat_data, aoi, quality_band= 'NDVI', calculate_coverage=False) #REPLACE OLD CODE WITH THE NEW ONE HERE
#Alternatively you can use temporal aggregation (ee reducer) to create mode cloudless imagery
#Add new functionality to calculate the coverage of the composite
median_landsat, coverage = composite.get_temporal_composite(landsat_data, aoi, reducer='Median', calculate_coverage=True) #REPLACE OLD CODE WITH THE NEW ONE HERE
#visualization parameter
l8_sr_visparam = {'min': 0,'max': 0.4,'gamma': [0.95, 1.1, 1],'bands':['NIR', 'RED', 'GREEN']}
#Add the data to the map
Map = geemap.Map()
Map.addLayer(mosaic_landsat, l8_sr_visparam, 'L8 SR Mosaic')
Map.addLayer(median_landsat, l8_sr_visparam, 'L8 SR Median')
Map.addLayer(landsat_data, l8_sr_visparam, 'L8 SR Image Collection')
# set center of the map in the area of interest
Map.centerObject(aoi, 7)

#retive thermal bands from TOA
thermal_bands, thermal_stats = optical_reflectance.get_thermal_bands(aoi, start, end, cloud_cover=40, compute_detailed_stats=False)
median_thermal = composite.get_temporal_composite(thermal_bands, aoi, reducer='Median') #REPLACE THE OLD CODE WITH THE NEW ONE
thermal_vis = {'min': 286,'max': 300,'gammma': 0.4}
#stacked all landsat bands and convert to float(making sure all data type are compatible)
stacked_landsat = median_landsat.addBands(median_thermal).toFloat()
#visualize the thermal bands and multispectral bands
Map.addLayer(median_thermal, thermal_vis, "Thermal Bands")
#Map

# Step 1a: Identify elements and properties
This step will identify the elements and properties that were defined in Step 0.
This step will be skipped in the testing and the identification will be done manually

# Step 1b: Classification Ruleset

Define all the functions to turn class definition into rulesets

In [ ]:
from luma_ge.modular_workflow import load_scheme

scheme1_rules = load_scheme("../data/test_data/scheme1_threshold.csv")
scheme2_rules = load_scheme("../data/test_data/scheme2_threshold.csv")

print(scheme2_rules[["class_id", "class_name", "rule", "priority"]].to_string())
print(scheme1_rules[["class_id", "class_name", "rule", "priority"]].to_string())

# Step 2: Build modular reference data

This step uploads the reference data that contains UML information inside its attribute table

In [ ]:
from luma_ge.modular_workflow import load_modular_training_data

data = load_modular_training_data(
    shp_path="../data/test_data/training_points.shp",
    aoi=aoi
)

training_gdf = data["gdf"]

training_fc = data["ee_fc"]

print(data["columns"])
print(data["size"])

# Step 3a: Apply classification ruleset to training dataset

In [ ]:
from luma_ge.modular_workflow import TrainingDataLabeller

labeller_scheme1 = TrainingDataLabeller(
    rules_df=scheme1_rules,
    scheme_name="my_scheme",
    nodata_value=0
)

scheme1_fc = labeller_scheme1.label(training_fc)

labeller_scheme2 = TrainingDataLabeller(
    rules_df=scheme2_rules,
    scheme_name="my_scheme",
    nodata_value=0
)

scheme2_fc = labeller_scheme2.label(training_fc)

In [ ]:
import geemap
import ee

m = geemap.Map()
m.centerObject(aoi, 12)

# Color palette (index = class_id)
palette = [
    "888780",  # 0 nodata
    "1D9E75",  # 1
    "378ADD",  # 2
    "D85A30",  # 3
    "BA7517",  # 4
    "7F77DD",  # 5
    "639922",  # 6
]

# --- Scheme 1 styling ---
scheme1_styled = scheme1_fc.map(
    lambda f: f.set({
        "style": {
            "color": "black",
            "pointSize": 5,
            "fillColor": ee.List(palette).get(
                ee.Number(f.get("class_id")).int()
            )
        }
    })
)

# --- Scheme 2 styling ---
scheme2_styled = scheme2_fc.map(
    lambda f: f.set({
        "style": {
            "color": "black",
            "pointSize": 5,
            "fillColor": ee.List(palette).get(
                ee.Number(f.get("class_id")).int()
            )
        }
    })
)

# --- Add layers ---
m.addLayer(aoi, {}, "AOI")

m.addLayer(
    scheme1_styled.style(**{"styleProperty": "style"}),
    {},
    "Scheme 1 (class_id)"
)

m.addLayer(
    scheme2_styled.style(**{"styleProperty": "style"}),
    {},
    "Scheme 2 (class_id)"
)

# --- Build legend from rules_df ---
# Use scheme1_rules (or whichever rules correspond)
legend_dict = {"0": "nodata"}

for _, row in scheme1_rules.iterrows():
    legend_dict[str(int(row["class_id"]))] = row["class_name"]

# Match colors to class IDs
legend_colors = palette[:len(legend_dict)]

# Add legend
m.add_legend(
    title="Class ID",
    legend_dict=legend_dict,
    colors=legend_colors
)

m

# Step 3b: Generate primitive layers

## Deterministic primitive layers

In [ ]:
from luma_ge.modular_workflow import PrimitiveLayerTrainer
# Generate primitive layers

# predictor image (from your previous step)
image = stacked_landsat

# modular training data
roi = training_fc

trainer = PrimitiveLayerTrainer(
    image=image,
    roi=roi
)

primitive_layers = trainer.train_all()

print(primitive_layers.keys())

### Visualize determinisic primitives

In [ ]:
import geemap

m = geemap.Map()

m.centerObject(aoi, 12)

m.addLayer(aoi, {}, "AOI")

m.addLayer(
    training_fc,
    {"color": "red"},
    "Training points"
)

m.addLayer(
    primitive_layers["tree_pres"],
    {"min": 0, "max": 1, "palette": ["white", "green"]},
    "tree_pres"
)
m.addLayer(
    primitive_layers["buil_pres"],
    {"min": 0, "max": 1, "palette": ["white", "orange"]},
    "buil_pres"
)
m.addLayer(
    primitive_layers["water_pres"],
    {"min": 0, "max": 1, "palette": ["white", "blue"]},
    "water_pres"
)

m

## Probabilistic primitive layers

In [ ]:
# Generate primitive layers

trainer = PrimitiveLayerTrainer(
    image=image,
    roi=roi
)

primitive_layers_mc = trainer.train_all_mc()

print(primitive_layers_mc.keys())

### Visualize probabilistic primitives

In [ ]:
# Visualize probabilistic primitive layers

m = geemap.Map()

m.centerObject(aoi, 12)

m.addLayer(aoi, {}, "AOI")


m.addLayer(
    training_fc,
    {"color": "red"},
    "Training points"
)

m.addLayer(
    primitive_layers_mc["tree_pres"],
    {"min": 0, "max": 1, "palette": ["white", "green"]},
    "tree_pres"
)
m.addLayer(
    primitive_layers_mc["buil_pres"],
    {"min": 0, "max": 1, "palette": ["white", "orange"]},
    "buil_pres"
)
m.addLayer(
    primitive_layers_mc["water_pres"],
    {"min": 0, "max": 1, "palette": ["white", "blue"]},
    "water_pres"
)

m

In [ ]:
# 1. Stack dict → ee.Image
primitive_stack_mc = ee.Image.cat(list(primitive_layers_mc.values()))
band_names = primitive_stack_mc.bandNames().getInfo()
print("Bands in probabilistic stack:", band_names)

# 2. Centroid needs a maxError argument when geometry comes from a shapefile
test_point = aoi.geometry().centroid(maxError=1)

# 3. Sample one pixel to verify probability output
sample = primitive_stack_mc.sample(
    region=test_point,
    scale=30,
    numPixels=1
).first().toDictionary().getInfo()

# Sample a subset of pixels inside AOI (adjust numPixels as needed)
sample_fc = primitive_stack_mc.sample(
    region=aoi,
    scale=30,
    numPixels=5000,   # or more, but keep it reasonable
    geometries=False
)

# Bring sampled data to client as a list of dicts
samples = sample_fc.getInfo()["features"]

# Convert to per‑band arrays
vals = {b: [] for b in band_names}
for f in samples:
    d = f["properties"]
    for b in band_names:
        vals[b].append(d[b])

print("Sample pixel values:")
for band, val in sample.items():
    status = "OK" if 0 < val < 1 else "BINARY — not probability!"
    print(f"  {band}: {val:.4f}  {status}")


import matplotlib.pyplot as plt
import numpy as np
import ee

# Convert to per‑band arrays
vals = {b: [] for b in band_names}
for f in samples:
    d = f["properties"]
    for b in band_names:
        vals[b].append(d[b])

# Plot histograms of probability values for each primitive
fig, axes = plt.subplots(1, len(band_names), figsize=(5*len(band_names), 4))
if len(band_names) == 1:
    axes = [axes]

for ax, b in zip(axes, band_names):
    arr = np.array(vals[b])

    # ---- statistics ----
    mean = np.mean(arr)
    std = np.std(arr)
    minv = np.min(arr)
    maxv = np.max(arr)
    median = np.median(arr)

    # ---- histogram ----
    ax.hist(arr, bins=20, range=(0, 1), color="steelblue", edgecolor="black")

    ax.set_title(b)
    ax.set_xlabel("Probability")
    ax.set_ylabel("Pixel count")

    # ---- show stats on plot ----
    text = (
        f"mean={mean:.3f}\n"
        f"std={std:.3f}\n"
        f"median={median:.3f}\n"
        f"min={minv:.3f}\n"
        f"max={maxv:.3f}"
    )

    ax.text(
        0.98, 0.98,
        text,
        transform=ax.transAxes,
        verticalalignment="top",
        horizontalalignment="right",
        bbox=dict(facecolor="white", alpha=0.8)
    )

        # ---- also print stats to console ----
    print(f"Statistics for {b}:")
    print(f"  Mean: {mean:.3f}")
    print(f"  Std: {std:.3f}")
    print(f"  Median: {median:.3f}")
    print(f"  Min: {minv:.3f}")
    print(f"  Max: {maxv:.3f}\n")

plt.tight_layout()
plt.show()

## Optional: clean up metadata and get the elements and properties

In [ ]:
# Clean each primitive layer manually, removing all properties including system:index
primitive_layers_clean = {}
for k, v in primitive_layers.items():
    img = ee.Image(v).toFloat().rename(k).copyProperties(v, [])  # copy no properties
    primitive_layers_clean[k] = img

# Concatenate into a single image
primitive_image = ee.Image.cat(list(primitive_layers_clean.values()))

# List layer names for reference
# Original keys
layer_names = list(primitive_layers.keys())

# Manually remove "system:index" if it exists
if "system:index" in layer_names:
    layer_names.remove("system:index")

print(layer_names)

# Step 4: Classification

## Generate using deterministic ruleset and deterministic layer

In [ ]:
from luma_ge.modular_workflow import RuleSetClassifier

scheme1_rules["scheme"] = "scheme1"  # add scheme column
scheme2_rules["scheme"] = "scheme2"  # add scheme column

# Initialize classifier
classifier1 = RuleSetClassifier(
    primitive_image=primitive_image,
    rules_df=scheme1_rules,
    aoi=aoi
)
classifier2 = RuleSetClassifier(
    primitive_image=primitive_image,
    rules_df=scheme2_rules,
    aoi=aoi
)


In [ ]:

# Deterministic classification
map1_det = classifier1.classify_scheme_deterministic("scheme1")
map2_det = classifier2.classify_scheme_deterministic("scheme2")

In [ ]:
from luma_ge.modular_workflow import validate_deterministic

# Step 1: validate deterministic results and pull as numpy
det_map1 = validate_deterministic(
    det_ee_image = map1_det,
    rules_df     = scheme1_rules,
    scheme_name  = "scheme1",
    aoi          = aoi,
    scale        = 30,
    scheme_label = "Scheme 1",
)

det_map2 = validate_deterministic(
    det_ee_image = map2_det,
    rules_df     = scheme2_rules,
    scheme_name  = "scheme2",
    aoi          = aoi,
    scale        = 30,
    scheme_label = "Scheme 2",
)

## Generate using Monte Carlo simulation and probabilistic layers

In [ ]:
classifier1_mc = RuleSetClassifier(
    primitive_image=primitive_stack_mc,   # stacked ee.Image, not the dict
    rules_df=scheme1_rules,
    aoi=aoi
)
classifier2_mc = RuleSetClassifier(
    primitive_image=primitive_stack_mc,
    rules_df=scheme2_rules,
    aoi=aoi
)

In [ ]:
results1 = classifier1_mc.classify_scheme_monte_carlo(
    scheme_name="scheme1",
    n_iterations=300,
    seed=42,
    scale=30,
)
 
results2 = classifier2_mc.classify_scheme_monte_carlo(
    scheme_name="scheme2",
    n_iterations=300,
    seed=42,
    scale=30,
)

In [ ]:
# Unpack MC outputs
map1_mode    = results1["mode_map"]
map1_entropy = results1["entropy_map"]
map1_probs   = results1["class_probs"]
 
map2_mode    = results2["mode_map"]
map2_entropy = results2["entropy_map"]
map2_probs   = results2["class_probs"]
 

In [ ]:
from luma_ge.modular_workflow import validate_monte_carlo

# Run validation for both schemes
validate_monte_carlo(results1, scheme1_rules, "scheme1",
                     entropy_threshold=0.5, scheme_label="Scheme 1")
validate_monte_carlo(results2, scheme2_rules, "scheme2",
                     entropy_threshold=0.5, scheme_label="Scheme 2")



In [ ]:
# Check what rules were actually generated for scheme 2
print(scheme2_rules[["class_id", "class_name", "rule", "priority"]].to_string())
print(scheme1_rules[["class_id", "class_name", "rule", "priority"]].to_string())
print(scheme2_rules[["class_id", "class_name", "rule", "priority"]].to_string())
print(scheme1_rules[["class_id", "class_name", "rule", "priority"]].to_string())

# Check what threshold values are in the rules
# A rule like "tree_pres > 0.7" will miss pixels where tree_pres=0.65
# even though the pixel is clearly forested

# Sample the primitive values at nodata pixels to understand what's there
band_arrays = classifier2_mc._get_band_arrays(scale=30)
nodata_mask = (results2["mode_map"] == 0)

# print("Primitive values at nodata pixels:")
# for name, arr in band_arrays.items():
#     vals = arr[nodata_mask]
#     print(f"  {name}: mean={vals.mean():.3f}  "
#           f"min={vals.min():.3f}  max={vals.max():.3f}")

## Compare deterministic vs mc

In [ ]:
from luma_ge.modular_workflow import compare_det_vs_mc

# Step 2: compare deterministic vs MC side by side
compare_det_vs_mc(
    det_class_map = det_map1,
    mc_results    = results1,
    rules_df      = scheme1_rules,
    scheme_name   = "scheme1",
    scheme_label  = "Scheme 1",
)

compare_det_vs_mc(
    det_class_map = det_map2,
    mc_results    = results2,
    rules_df      = scheme2_rules,
    scheme_name   = "scheme2",
    scheme_label  = "Scheme 2",
)

In [ ]:
# If most values are between 0.3 and 0.7, the RF is uncertain everywhere
# and both methods are guessing rather than classifying
for name, arr in band_arrays.items():
    mid = ((arr > 0.3) & (arr < 0.7)).mean() * 100
    print(f"{name}: {mid:.1f}% of pixels in uncertain zone [0.3, 0.7]")

## Generate using class-labelled training dataset (Step 3a)

In [ ]:
# Sample the image at the labelled points
sample = stacked_landsat.sampleRegions(
    collection=scheme1_fc,
    properties=["class_id"],  # use the labelled class_id
    scale=30,
    geometries=False
)

# Create a Random Forest classifier with 50 trees
classifier = ee.Classifier.smileRandomForest(50)

# Train the classifier on the sampled points
trained_classifier = classifier.train(
    features=sample,
    classProperty="class_id",
    inputProperties= stacked_landsat.bandNames()
)

# Classify the image
classified_image = stacked_landsat.classify(trained_classifier)

# Rename the output band to class_id for clarity
classified_image = classified_image.rename("class_id")

# Display a quick summary
print("Classification complete.")

In [ ]:
from luma_ge.modular_workflow import validate_deterministic

# Step 1: validate deterministic results and pull as numpy
det_map1 = validate_deterministic(
    det_ee_image = classified_image,
    rules_df     = scheme1_rules,
    scheme_name  = "scheme1",
    aoi          = aoi,
    scale        = 30,
    scheme_label = "Scheme 1",
)